<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python for Finance, 3rd Edition
## Appendix B · Probability, Statistics, and Stochastic Calculus Essentials

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by various LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Notebook Goals
This notebook mirrors the appendix examples in a Colab-ready format so that you can run, tweak, and extend them interactively.

### How to Use This Notebook
- Run the cells top to bottom the first time to create all variables.
- Use additional cells for your own experiments or GenAI-assisted refactorings.
- Refer back to the book text for detailed explanations and context.

In [ ]:
from pathlib import Path
import subprocess
import sys

NOTEBOOK_SUBDIR = "notebooks"
COLAB_PACKAGES = {}
REPO_NAME = "py4fi3rd"
REPO_URL = "https://github.com/yhilpisch/py4fi3rd.git"


def _support_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        support_dir = candidate / "notebooks"
        if (support_dir / "_book_notebook_support.py").exists():
            return support_dir
    if "google.colab" in sys.modules:
        root = Path("/content") / REPO_NAME
        if not root.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(root)],
                check=True,
            )
        return root / "notebooks"
    raise RuntimeError("Could not locate notebook support helpers.")


SUPPORT_DIR = _support_dir()
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))

from _book_notebook_support import setup_notebook

CONTEXT = setup_notebook(
    notebook_subdir=NOTEBOOK_SUBDIR,
    colab_packages=COLAB_PACKAGES,
)

PROJECT_ROOT = CONTEXT["PROJECT_ROOT"]
NOTEBOOK_DIR = CONTEXT["NOTEBOOK_DIR"]
CODE_DIR = CONTEXT["CODE_DIR"]
CHAPTERS_DIR = CONTEXT["CHAPTERS_DIR"]
FIGURES_DIR = CONTEXT["FIGURES_DIR"]
DATA_DIR = CONTEXT["DATA_DIR"]

PROJECT_ROOT

## Random Variables and Distributions
A random variable maps uncertain outcomes to numbers.

In [ ]:
from scipy import stats

In [ ]:
dist = stats.norm(loc=0.08, scale=0.20)

In [ ]:
round(dist.ppf(0.05), 4), round(dist.cdf(-0.10), 4)

## Expectations, Variance, and Sampling Error
The expectation latexmath:[\mathbb{E}(X)] is the probability-weighted average
of a random variable.

In [ ]:
import numpy as np

In [ ]:
rng = np.random.default_rng(42)

In [ ]:
rets = rng.normal(0.0004, 0.012, size=252)

In [ ]:
ann_mean = float(rets.mean() * 252)

In [ ]:
ann_vol = float(rets.std(ddof=1) * np.sqrt(252))

In [ ]:
round(ann_mean, 4), round(ann_vol, 4)

In [ ]:
round(float(stats.sem(rets) * 252), 4)

## The Central Limit Theorem
The central limit theorem says that averages of many independent, identically
distributed observations are approximately normal under broad conditions.

In [ ]:
rng = np.random.default_rng(7)

In [ ]:
samples = rng.normal(0.0004, 0.012, size=(10_000, 252))

In [ ]:
means = samples.mean(axis=1)

In [ ]:
scaled = (means - 0.0004) / (0.012 / np.sqrt(252))

In [ ]:
np.quantile(scaled, [0.025, 0.5, 0.975]).round(3)

## Covariance and Correlation
Covariance measures how two random variables move together.

In [ ]:
prices = np.array([
    [100, 100],
    [102, 99],
    [101, 101],
    [105, 103],
], dtype=float)

In [ ]:
log_rets = np.diff(np.log(prices), axis=0)

In [ ]:
np.cov(log_rets, rowvar=False).round(6)

In [ ]:
np.corrcoef(log_rets, rowvar=False).round(4)

## Maximum Likelihood Estimation
Maximum likelihood estimation chooses parameters that make the observed data as
probable as possible under a chosen model.

In [ ]:
mu_hat = float(rets.mean())

In [ ]:
sigma_hat = float(rets.std(ddof=0))

In [ ]:
ll = np.sum(stats.norm.logpdf(rets, loc=mu_hat, scale=sigma_hat))

In [ ]:
round(mu_hat, 6), round(sigma_hat, 6), round(float(ll), 2)

## Brownian Motion
Brownian motion latexmath:[W_t] is a continuous-time stochastic process with
three defining properties: it starts at zero, has independent increments, and
has normally distributed increments latexmath:[W_t - W_s \sim N(0, t-s)] for
latexmath:[0 \leq s < t].

In [ ]:
rng = np.random.default_rng(11)

In [ ]:
dt = 1 / 252

In [ ]:
increments = rng.normal(0.0, np.sqrt(dt), size=252)

In [ ]:
brownian = np.cumsum(increments)

In [ ]:
round(float(brownian[-1]), 4), round(float(np.var(increments, ddof=1)), 6)

## Ito's Lemma
Ito's lemma is the chain rule for stochastic processes.

In [ ]:
import sympy as sy

In [ ]:
S, mu, sigma = sy.symbols("S mu sigma", positive=True)

In [ ]:
f = sy.log(S)

In [ ]:
drift = mu * S * sy.diff(f, S)

In [ ]:
drift += sy.Rational(1, 2) * sigma**2 * S**2 * sy.diff(f, S, 2)

In [ ]:
diffusion = sigma * S * sy.diff(f, S)

In [ ]:
sy.simplify(drift), sy.simplify(diffusion)

## Geometric Brownian Motion
Geometric Brownian motion (GBM) is defined by latexmath:[dS_t = \mu S_t dt +
\sigma S_t dW_t].

In [ ]:
rng = np.random.default_rng(12)

In [ ]:
s0, r, sigma, T = 100.0, 0.03, 0.20, 1.0

In [ ]:
z = rng.standard_normal(200_000)

In [ ]:
st = s0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * z)

In [ ]:
round(float(st.mean()), 3), round(float(s0 * np.exp(r * T)), 3)

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">